In [ ]:
import random
import sys
import numpy as np
import torch

from pathlib import Path
from pycocotools.coco import COCO
from torch.utils.data import DataLoader

PROJECT_ROOT = Path("/kaggle/working")
INPUT_ROOT = Path("/kaggle/input")
COMPETITION = "multiple-object-video-detection-hw-2-ucu-ai-2026"

DIRECTORY = INPUT_ROOT / COMPETITION
if not (DIRECTORY / "train" / "annotations_coco.json").is_file():
    data_roots = sorted({path.parent.parent
                         for path in INPUT_ROOT.rglob("annotations_coco.json")
                         if path.parent.name == "train"})
    if len(data_roots) != 1:
        raise FileNotFoundError(
            "Attach the extracted competition data, or set DIRECTORY to its input folder."
        )
    DIRECTORY = data_roots[0]

INFO_TRAIN = DIRECTORY / "train" / "annotations_coco.json"
INFO_VAL = DIRECTORY / "val" / "annotations_coco.json"
INFO_TEST = DIRECTORY / "test_images.csv"

required_scripts = {
    "data_prep.py",
    "data_cv.py",
    "dataset_helpers.py",
    "torch_model.py",
    "yolo26_model.py",
    "experiment_runner.py",
    "experiment_results.py",
}

script_dirs = sorted({path.parent for path in INPUT_ROOT.rglob("yolo26_model.py")
    if path.parent.name == "scripts" and all((path.parent / filename).is_file() for filename in required_scripts)
})
scripts_parent = str(script_dirs[0].parent)

if scripts_parent not in sys.path:
    sys.path.insert(0, scripts_parent)

from scripts.data_prep import load_annotations, save_fold_yaml, load_yaml
from scripts.data_cv import stratified_group_folds
from scripts.dataset_helpers import DetectionDataset, collate_data
from scripts.experiment_runner import run_yolo_cv
from scripts.experiment_results import model_training_results_yolo_cv

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SEED = 42
FOLDS = 3
EPOCHS = 5
BATCH_SIZE = 8
FRAME_STEP = 15

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
train_frames, train_boxes = load_annotations(COCO(str(INFO_TRAIN)), split="train")
frames = stratified_group_folds(train_frames, train_boxes, n_splits=FOLDS, seed=SEED)
annotations_by_image = {image_id: group for image_id, group in train_boxes.groupby("image_id")}
empty_annotations = train_boxes.iloc[:0].copy()

loading annotations into memory...
Done (t=6.30s)
creating index...
index created!


In [ ]:
cv_fold_loaders, fold_yaml_paths = {}, {}

yolo_dir = (DIRECTORY / "train").resolve()
class_names = load_yaml(yolo_dir / "data.yaml")["names"]

fold_output_dir = PROJECT_ROOT / "yolo_cv"
folds_dir = fold_output_dir / "folds"
folds_dir.mkdir(parents=True, exist_ok=True)

kwargs_dataset = {
    "directory": DIRECTORY,
    "annotations_by_image": annotations_by_image,
    "empty_annotations": empty_annotations,
}

for fold in range(FOLDS):
    training_frames = (frames.loc[frames[f"train_fold_{fold}"].eq(True)].sort_values(["video", "frame_id"])
        .groupby("video", group_keys=False).nth[::FRAME_STEP].reset_index(drop=True))

    validation_frames = frames.loc[frames["validation_fold"].eq(fold)].reset_index(drop=True)

    for split, split_frames in [("train", training_frames), ("val", validation_frames),]:
        image_paths = [str((DIRECTORY / frame["split"] / frame["file_name"]).resolve())
            for _, frame in split_frames.iterrows()
        ]

        (folds_dir / f"fold_{fold}_{split}.txt").write_text("\n".join(image_paths) + "\n", encoding="utf-8")

    fold_yaml_paths[fold] = save_fold_yaml(dataset_dir=yolo_dir, output_dir=fold_output_dir, fold=fold, class_names=class_names)
    valid_dataset = DetectionDataset(frames=validation_frames, validation=True, **kwargs_dataset)

    cv_fold_loaders[fold] = {
        "valid": DataLoader(
            valid_dataset,
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=0,
            collate_fn=collate_data
        )
    }

    print(f"Fold {fold}: {len(training_frames)} training frames, {len(validation_frames)} validation frames")

Fold 0: 1107 training frames, 7943 validation frames
Fold 1: 1094 training frames, 8088 validation frames
Fold 2: 1091 training frames, 8170 validation frames


In [ ]:
cv_kwargs = {"cv_fold_loaders": cv_fold_loaders, "fold_yaml_paths": fold_yaml_paths,
             "annotations_path": INFO_TRAIN, "device": device, "seed": SEED,
             "epochs": EPOCHS, "batch_size": BATCH_SIZE, "lr": 1e-4, 
             "weight_decay": 1e-4, "optimizer": "Adam", "validate_each_epoch": False}
fold_dirs = {fold: f"fold_{fold}" for fold in range(FOLDS)}

In [ ]:
baseline_histories, baseline_results = run_yolo_cv(**cv_kwargs, output_dir=PROJECT_ROOT / "outputs" / "yolo26_baseline_v2",
                                                   imgsz=640, box_weight=7.5)

loading annotations into memory...
Done (t=5.10s)
creating index...
index created!
Ultralytics 8.4.203 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/yolo_cv/fold_0.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=5, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosa

/usr/local/lib/python3.13/dist-packages/ray/train/_internal/session.py:683: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(


        2/5      1.61G      2.297      1.967   0.003931         62        640: 100% ━━━━━━━━━━━━ 139/139 7.7it/s 18.1s0.3s
Loading and preparing results...
DONE (t=0.00s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.06s).
Accumulating evaluation results...
DONE (t=0.01s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.169
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.302
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.177
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.024
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.196
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.550
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.014
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.411
 Average Recall     (AR) 

In [ ]:
baseline_results_wrap = model_training_results_yolo_cv(PROJECT_ROOT, filename="yolo26_baseline_v2", fold_dirs=fold_dirs)


Fold 0


,epoch,train_loss,learning_rate,mAP,AP50,AP_small
0,1,4.754469,0.000033,0.057090,0.128353,0.028058
1,2,4.268090,0.000053,0.169115,0.302052,0.023789
2,3,4.114877,0.000060,0.178708,0.289945,0.042250
3,4,4.063425,0.000041,0.220760,0.351481,0.040062
4,5,4.028758,0.000021,0.223719,0.350249,0.040323



Fold 1


,epoch,train_loss,learning_rate,mAP,AP50,AP_small
0,1,5.015748,0.000033,0.146953,0.261779,0.018743
1,2,4.454722,0.000053,0.193823,0.345424,0.047666
2,3,4.366423,0.000060,0.234198,0.417235,0.043850
3,4,4.275827,0.000041,0.234047,0.423420,0.048207
4,5,4.257132,0.000021,0.234760,0.433599,0.045831



Fold 2


,epoch,train_loss,learning_rate,mAP,AP50,AP_small
0,1,4.948909,0.000033,0.065493,0.161757,0.051290
1,2,4.378680,0.000053,0.078245,0.168286,0.050954
2,3,4.279161,0.000060,0.122964,0.296599,0.093878
3,4,4.195267,0.000041,0.151976,0.308072,0.103123
4,5,4.157253,0.000021,0.146036,0.292138,0.110891



Best results per fold


,fold,epoch,train_loss,learning_rate,mAP,AP50,AP_small
0,0,5,4.028758,0.000021,0.223719,0.350249,0.040323
1,1,5,4.257132,0.000021,0.234760,0.433599,0.045831
2,2,4,4.195267,0.000041,0.151976,0.308072,0.103123



Metrics mean and std across folds


,mAP,AP50,AP_small
mean,0.203485,0.363973,0.063092
std,0.044948,0.063879,0.034777


In [ ]:
resolution_histories, resolution_results = run_yolo_cv(**cv_kwargs, output_dir=PROJECT_ROOT / "outputs" / "yolo26_res1024", imgsz=1024, box_weight=7.5)

loading annotations into memory...
Done (t=5.25s)
creating index...
index created!
Ultralytics 8.4.203 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/yolo_cv/fold_0.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=5, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, iou=0.7, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mos

In [ ]:
resolution_results_wrap =  model_training_results_yolo_cv(PROJECT_ROOT, "yolo26_res1024", fold_dirs)


Fold 0


,epoch,train_loss,learning_rate,mAP,AP50,AP_small
0,1,4.637714,0.000033,0.064455,0.187413,0.024187
1,2,4.095200,0.000053,0.164623,0.313456,0.030239
2,3,3.943300,0.000060,0.161954,0.358013,0.035939
3,4,3.891425,0.000041,0.174007,0.364069,0.035859
4,5,3.859785,0.000021,0.174658,0.345151,0.033392



Fold 1


,epoch,train_loss,learning_rate,mAP,AP50,AP_small
0,1,4.926278,0.000033,0.134578,0.238204,0.054220
1,2,4.271822,0.000053,0.199812,0.317082,0.100767
2,3,4.192681,0.000060,0.226403,0.354999,0.113564
3,4,4.109623,0.000041,0.248982,0.412334,0.073860
4,5,4.074870,0.000021,0.254460,0.419663,0.099267



Fold 2


,epoch,train_loss,learning_rate,mAP,AP50,AP_small
0,1,4.810547,0.000033,0.094077,0.173424,0.084496
1,2,4.231909,0.000053,0.088767,0.181170,0.081364
2,3,4.126069,0.000060,0.119199,0.321761,0.082782
3,4,4.034190,0.000041,0.149135,0.309486,0.093439
4,5,4.007069,0.000021,0.100416,0.183359,0.092062



Best results per fold


,fold,epoch,train_loss,learning_rate,mAP,AP50,AP_small
0,0,3,3.726113,0.000060,0.196619,0.391438,0.055733
1,1,5,3.874518,0.000021,0.328495,0.560132,0.089963
2,2,1,4.749170,0.000033,0.176921,0.314882,0.254083



Metrics mean and std across folds


,mAP,AP50,AP_small
mean,0.234012,0.422151,0.133260
std,0.082415,0.125477,0.106026


In [ ]:
box_histories, box_results = run_yolo_cv(**cv_kwargs, output_dir=PROJECT_ROOT / "outputs" / "yolo26_box", imgsz=1024, box_weight=10.0)

loading annotations into memory...
Done (t=5.17s)
creating index...
index created!
Ultralytics 8.4.203 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=10.0, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/yolo_cv/fold_0.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=5, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, iou=0.7, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mo

In [ ]:
box_results_wrap = model_training_results_yolo_cv(PROJECT_ROOT, "yolo26_box", fold_dirs)


Fold 0


,epoch,train_loss,learning_rate,mAP,AP50,AP_small
0,1,4.754469,0.000033,0.057090,0.128353,0.028058
1,2,4.268090,0.000053,0.169115,0.302052,0.023789
2,3,4.114877,0.000060,0.178708,0.289945,0.042250
3,4,4.063425,0.000041,0.220760,0.351481,0.040062
4,5,4.028758,0.000021,0.223719,0.350249,0.040323



Fold 1


,epoch,train_loss,learning_rate,mAP,AP50,AP_small
0,1,5.015748,0.000033,0.146953,0.261779,0.018743
1,2,4.454722,0.000053,0.193823,0.345424,0.047666
2,3,4.366423,0.000060,0.234198,0.417235,0.043850
3,4,4.275827,0.000041,0.234047,0.423420,0.048207
4,5,4.257132,0.000021,0.234760,0.433599,0.045831



Fold 2


,epoch,train_loss,learning_rate,mAP,AP50,AP_small
0,1,4.948909,0.000033,0.065493,0.161757,0.051290
1,2,4.378680,0.000053,0.078245,0.168286,0.050954
2,3,4.279161,0.000060,0.122964,0.296599,0.093878
3,4,4.195267,0.000041,0.151976,0.308072,0.103123
4,5,4.157253,0.000021,0.146036,0.292138,0.110891



Best results per fold


,fold,epoch,train_loss,learning_rate,mAP,AP50,AP_small
0,0,5,4.300518,0.000021,0.279655,0.753132,0.420435
1,1,4,4.579551,0.000041,0.329890,0.552645,0.095427
2,2,2,4.751470,0.000053,0.147839,0.319330,0.181485



Metrics mean and std across folds


,mAP,AP50,AP_small
mean,0.252461,0.541702,0.232449
std,0.094022,0.217108,0.168391
